In [ ]:
import ee
import pandas as pd
import datetime

# 1. Initialize GEE
ee.Authenticate()
# ee.Initialize(project='tp-vae-wgan')
ee.Initialize(project='tp-vae-wgan', opt_url='https://earthengine-highvolume.googleapis.com')

## Clean Dataset 

In [ ]:
import numpy as np
import os
import concurrent.futures
import io
import requests

def get_sharpest_clean_image(poi, start_date, end_date):
    """
    Acquiert l'image la plus nette (single-shot) pour préserver les textures 
    nécessaires au WGAN.
    """
    # 1. Collection Sentinel-2 Harmonized
    s2_col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
               .filterBounds(poi) \
               .filterDate(start_date, end_date) \
               .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))

    # 2. On prend la SEULE meilleure image au lieu de faire une médiane
    # Cela évite le flou de registration et préserve les textures haute-fréquence
    sharp_img = s2_col.sort('CLOUDY_PIXEL_PERCENTAGE').first()
    
    # 3. Resampling bicubique pour aligner proprement les grilles 10m et 20m
    # Note: On garde les valeurs RAW (0-10000) comme demandé
    return sharp_img.resample('bicubic').select(['B4', 'B3', 'B2', 'B12'])

def download_patch(args):
    poi, name, idx, start, end, folder, valid_stack = args
    try:
        # Vérification Forestière & Historique Incendie (One-shot)
        stats = ee.Image.cat([valid_stack['land_cover'], valid_stack['burned_area']]) \
                      .reduceRegion(ee.Reducer.first(), poi, 20).getInfo()
        
        if stats.get('LC_Type1') is None or not (1 <= stats.get('LC_Type1') <= 5): return False
        if stats.get('BurnDate') is not None and stats.get('BurnDate') > 0: return False

        # Acquisition de l'image la plus nette
        img = get_sharpest_clean_image(poi, start, end)
        region = poi.buffer(640).bounds() 
        
        # Téléchargement direct en format NPY
        url = img.getDownloadURL({
            'scale': 20, 
            'crs': 'EPSG:4326', 
            'region': region, 
            'format': 'NPY'
        })
        
        response = requests.get(url)
        if response.status_code == 200:
            data_raw = np.load(io.BytesIO(response.content))
            # Empilage des bandes sans division (valeurs brutes)
            data = np.stack([data_raw[b] for b in ['B4', 'B3', 'B2', 'B12']], axis=-1)
            
            if data.shape[0] >= 64 and data.shape[1] >= 64:
                # Sauvegarde en int16 ou float32 (valeurs brutes)
                np.save(f"{folder}/clean_{name}_{idx}.npy", data[:64, :64, :])
                return True
        return False
    except: return False

# Configuration des Régions
REGIONS = {
    "canada":    {"geo": ee.Geometry.Rectangle([-120, 50, -70, 60]), "months": [6, 8]},
    "morocco":   {"geo": ee.Geometry.Rectangle([-6.0, 34.0, -2.0, 36.0]), "months": [3, 6]},
    "spain":     {"geo": ee.Geometry.Rectangle([-9.0, 36.0, 3.0, 43.0]), "months": [4, 7]},
    "russia":    {"geo": ee.Geometry.Rectangle([30.0, 50.0, 130.0, 70.0]), "months": [6, 8]},
    "australia": {"geo": ee.Geometry.Rectangle([113.0, -43.0, 153.0, -10.0]), "months": [5, 9]},
    "brazil":    {"geo": ee.Geometry.Rectangle([-73.0, -33.0, -34.0, 5.0]), "months": [5, 10]}
}

def run_factory(samples_per_region=50):
    output_folder = './dataset/train_clean'
    if not os.path.exists(output_folder): os.makedirs(output_folder)
    
    valid_stack = {
        'land_cover': ee.Image("MODIS/061/MCD12Q1/2022_01_01").select('LC_Type1'),
        'burned_area': ee.ImageCollection("MODIS/061/MCD64A1").filterDate('2023-01-01', '2023-12-31').select('BurnDate').max()
    }
    
    tasks = []
    for name, config in REGIONS.items():
        points = ee.FeatureCollection.randomPoints(config['geo'], samples_per_region*12, seed=42).getInfo()['features']
        start = f"2023-{config['months'][0]:02d}-01"
        end = f"2023-{config['months'][1]:02d}-28"
        for i, pt in enumerate(points):
            tasks.append((ee.Geometry.Point(pt['geometry']['coordinates']), name, i, start, end, output_folder, valid_stack))

    print(f"Lancement de la production d'images nettes (Single-shot)...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
        results = list(executor.map(download_patch, tasks))
    
    print(f"Downloaded {sum(results)} patchs.")

run_factory(samples_per_region=200)

Lancement de la production d'images nettes (Single-shot)...
Downloaded 1395 patchs.


## Fire Dataset

In [ ]:
CSV_FILES = {
    "canada": "dataset/MODIS/modis_2024_Canada.csv",
    "brazil": "dataset/MODIS/modis_2024_Brazil.csv",
    "australia": "dataset/MODIS/modis_2024_Australia.csv",
    "spain": "dataset/MODIS/modis_2024_Spain.csv",
    "morocco": "dataset/MODIS/modis_2024_Morocco.csv"
}

def get_fire_image(poi, acq_date):
    """
    Fetches the sharpest Sentinel-2 image within 3 days of the MODIS fire detection.
    """
    # Ensure date is a string to prevent GEE parsing errors
    start_date = ee.Date(str(acq_date))
    end_date = start_date.advance(5, 'day')
    
    s2_col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
               .filterBounds(poi) \
               .filterDate(start_date, end_date) \
               .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))

    # Sort by cloud cover to get the clearest texture of the fire/scar
    fire_img = s2_col.sort('CLOUDY_PIXEL_PERCENTAGE').first()
    
    # Resample bicubic to align the 10m (RGB) and 20m (SWIR) bands to a 20m grid
    # We keep RAW values (0-10000) for custom normalization in the next cell
    return fire_img.resample('bicubic').select(['B4', 'B3', 'B2', 'B12'])

def download_fire_patch(args):
    lon, lat, acq_date, name, idx, folder, land_cover_img = args
    try:
        poi = ee.Geometry.Point([lon, lat])
        
        # 1. Forest Check: Verify this fire occurred in a forest (Classes 1-5)
        # This ensures the VAE is tested on 'Forest Anomaly' specifically
        lc = land_cover_img.reduceRegion(ee.Reducer.first(), poi, 20).get('LC_Type1').getInfo()
        if lc is None or not (1 <= lc <= 5): return False

        # 2. Acquisition of the sharpest available image
        img = get_fire_image(poi, acq_date)
        if img is None: return False
        
        # 64 pixels * 20m scale = 1280m patch
        region = poi.buffer(640).bounds() 
        
        url = img.getDownloadURL({
            'scale': 20, 
            'crs': 'EPSG:4326', 
            'region': region, 
            'format': 'NPY'
        })
        
        response = requests.get(url)
        if response.status_code == 200:
            data_raw = np.load(io.BytesIO(response.content))
            # Stack bands into (H, W, 4) order: Red, Green, Blue, SWIR
            data = np.stack([data_raw[b] for b in ['B4', 'B3', 'B2', 'B12']], axis=-1)
            
            if data.shape[0] >= 64 and data.shape[1] >= 64:
                # Save as fire_{region}_{idx}.npy (Raw Integers)
                np.save(f"{folder}/fire_{name}_{idx}.npy", data[:64, :64, :])
                return True
        return False
    except Exception:
        return False

def run_fire_factory(samples_per_region=200):
    output_folder = './dataset/test_fire'
    if not os.path.exists(output_folder): os.makedirs(output_folder)
    
    # Static Land Cover image used for all workers
    land_cover_img = ee.Image("MODIS/061/MCD12Q1/2022_01_01").select('LC_Type1')
    tasks = []
    
    for name, csv_path in CSV_FILES.items():
        if not os.path.exists(csv_path):
            print(f"Skipping {name}: {csv_path} not found.")
            continue
            
        print(f"Queueing CSV for {name}...")
        df = pd.read_csv(csv_path)
        
        # Filter for High Confidence (>90) NASA FIRMS detections
        df_clean = df[df['confidence'] > 90].copy()
        df_clean['acq_date'] = df_clean['acq_date'].astype(str)
        
        # Sample points to account for forest/cloud filter skips
        sample_size = min(len(df_clean), samples_per_region * 6)
        sample_df = df_clean.sample(n=sample_size) 
        
        for i, row in sample_df.iterrows():
            tasks.append((row['longitude'], row['latitude'], row['acq_date'], name, i, output_folder, land_cover_img))

    print(f"Starting parallel download of Fire Patches...")
    # 10 workers is the optimal balance for the High-Volume endpoint quotas
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        results = list(executor.map(download_fire_patch, tasks))
    
    print(f"downloaded {sum(results)} fire patches to {output_folder}.")

# Start the factory
run_fire_factory(samples_per_region=200)

Queueing CSV for canada...


Queueing CSV for brazil...
Queueing CSV for australia...
Queueing CSV for spain...
Queueing CSV for morocco...
Starting parallel download of Fire Patches...
downloaded 565 fire patches to ./dataset/test_fire.
